# Phase 2 — PDF parsing and chunk audit

**Goal:** Understand the PDF → full-page text/tables → chunks workflow and identify financial information that extraction can lose. This lesson does not download data, rebuild the parser/index, call the model or read test reference answers.

Phase 0 is complete, and you have authorised Phase 2. You may select **Run All** in this notebook using the **Financial RAG** kernel. At the end, stop and wait for confirmation before Phase 3.

1. Verify the existing 26 reports, page mappings and chunks.
2. Plot page/table counts, chunk lengths and warnings.
3. Re-extract 13 development evidence pages and save source-page images.
4. Use a Pfizer example to compare the PDF, page text, table and chunk.
5. Create an audit worksheet that distinguishes programmatic checks from manual content review.

All new outputs go to `rag_study/outputs/data_audit/phase2_line_aware_v2/`. The figures diagnose data quality; they do not measure model accuracy.

**21 October repair:** the active corpus uses line-aware chunks and table-admission checks. Steps 5–6 retain the original example for comparison; Step 6 then shows the repair. These checks do not rate answer correctness.

**Extraction-review update (22 October 2026):** at the student's request, a delegated AI agent reviewed all 13 source pages and current chunks. The worksheet records the review method explicitly. The author subsequently confirmed checking the 13 pages and agreeing with the findings; no generated-answer human ratings are required under the automatic protocol. Inspect `agent_review_20261022/agent_review.md` under the Phase 2 output folder for the rubric and page findings.

**Author confirmation:** on 22 October 2026 the student reported completing the 13-page human check and agreeing with the AI findings. The worksheet records this confirmation; its warning/no/uncertain ratings remain unchanged. This does not score generated answers.

In [ ]:
from pathlib import Path
import sys, json, hashlib
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rag_study/config.json").exists())
assert Path(sys.prefix).resolve() == (ROOT / "rag_study/.venv").resolve(), "Select Financial RAG."
sys.path.insert(0, str(ROOT / "rag_study"))
import study  # existing read/parse helpers; importing does not run the experiment
DATA = study.DATA
chunking_manifest = study.check_chunking_revision()
assert chunking_manifest["revision"] == "line_aware_v2"
OUT = ROOT / "rag_study/outputs/data_audit/phase2_line_aware_v2"
OUT.mkdir(parents=True, exist_ok=True)
phase1 = json.loads((ROOT / "rag_study/outputs/data_audit/phase1_verification.json").read_text())
assert phase1["pdf_hashes_verified"] and phase1["pdf_count"] == 26
manifest = pd.read_csv(DATA / "manifest.csv")
dev = study.read_jsonl(DATA / "dev.jsonl")  # development records only
pages = [p for f in sorted((DATA / "parsed").glob("*.jsonl")) for p in study.read_jsonl(f)]
chunks = study.read_jsonl(DATA / "chunks.jsonl")
warnings = study.read_jsonl(DATA / "parsing_failures.jsonl")
page_lookup = {(p["doc_name"], p["page0"]): p for p in pages}
print("Existing parser output loaded. No model calls or corpus rebuild.")


## Step 1 — Source mapping: Can each passage be traced to its source page?

`page0` is Python's zero-based index; `pdf_page` is the one-based page number shown in a PDF viewer. The printed page label may differ. The code checks uniqueness, consecutive page numbers and chunk IDs. It cannot prove that the financial content is correct.

The report statistics below include both development and evaluation documents, without inspecting evaluation questions or answers.

In [ ]:
assert len(page_lookup) == len(pages), "Duplicate page records"
assert len({c["id"] for c in chunks}) == len(chunks), "Duplicate chunk IDs"
assert set(p["doc_name"] for p in pages) == set(manifest["doc_name"])
for row in manifest.to_dict("records"):
    doc_pages = [p for p in pages if p["doc_name"] == row["doc_name"]]
    assert sorted(p["page0"] for p in doc_pages) == list(range(row["pdf_pages"]))
    assert all(p["pdf_page"] == p["page0"] + 1 for p in doc_pages)
for c in chunks:
    assert (c["doc_name"], c["page0"]) in page_lookup
    assert c["pdf_page"] == c["page0"] + 1
    assert c["id"].startswith(f'{c["doc_name"]}:p{c["pdf_page"]}:c')

page_df = pd.DataFrame([{ "doc_name": p["doc_name"], "page0": p["page0"], "pdf_page": p["pdf_page"],
    "text_characters": len(p["text"].strip()), "detected_tables": len(p["tables"]),
    "extraction_error": bool(p["errors"])} for p in pages])
chunk_df = pd.DataFrame([{k:c[k] for k in ["id", "doc_name", "page0", "pdf_page", "kind", "embedding_tokens"]} for c in chunks])
page_df.to_csv(OUT / "page_inventory.csv", index=False)
chunk_df.to_csv(OUT / "chunk_inventory.csv", index=False)
summary = {"pdfs": len(manifest), "parsed_pages": len(pages), "chunks": len(chunks),
    "text_chunks": int((chunk_df.kind == "text").sum()), "table_chunks": int((chunk_df.kind == "table").sum()),
    "blank_text_pages": int((page_df.text_characters == 0).sum()),
    "pages_with_extraction_errors": int(page_df.extraction_error.sum()),
    "recorded_warnings": len(warnings), "mapping_checks_passed": True}
display(pd.DataFrame(summary.items(), columns=["Check", "Observed value"]))


## Step 2 — Corpus figures: Document lengths and table detection

**Figure P2-1** shows each complete report's page count on the left and the number of pages with at least one detected table on the right. Failing to detect a table does not mean that a page contains no financial table. The colours are not quality ratings.

**Figure P2-2** illustrates the Pfizer page-number conversion. Formal citations use the PDF viewer page number; check the source page to identify its printed label.

In [ ]:
by_doc = page_df.groupby("doc_name").agg(pages=("page0", "size"),
    table_pages=("detected_tables", lambda x: int((x > 0).sum())), blank_pages=("text_characters", lambda x: int((x == 0).sum())))
by_doc = by_doc.sort_values("pages")
fig, axes = plt.subplots(1, 2, figsize=(13, 9), sharey=True)
axes[0].barh(by_doc.index, by_doc.pages, color="#42657d")
axes[1].barh(by_doc.index, by_doc.table_pages, color="#b87b40")
axes[0].set(xlabel="All parsed pages", title="Full-report search space")
axes[1].set(xlabel="Pages with detected tables", title="Table detection is not table accuracy")
fig.suptitle("P2-1: 26 reports — extraction inventory")
fig.tight_layout(); fig.savefig(OUT / "P2_1_report_inventory.png", dpi=170); plt.show()

fig, ax = plt.subplots(figsize=(10, 2.4)); ax.axis("off")
labels = ["FinanceBench annotation\nevidence_page_num = 58", "Python access\npdf.pages[58]", "PDF viewer\npage 59"]
for x, label in zip([0.17, 0.5, 0.83], labels):
    ax.text(x, .5, label, ha="center", va="center", fontsize=12,
            bbox={"boxstyle":"round,pad=0.8", "facecolor":"#edf2f5", "edgecolor":"#42657d"})
for a,b in [(.32,.37),(.65,.7)]: ax.annotate("", xy=(b,.5), xytext=(a,.5), arrowprops={"arrowstyle":"->"})
ax.set_title("P2-2: Pfizer FY2021 — one page, three references")
fig.tight_layout(); fig.savefig(OUT / "P2_2_page_mapping.png", dpi=170); plt.show()


## Step 3 — Chunk lengths: Length does not establish evidence completeness

Use the local BGE tokenizer to recount each chunk's tokens and compare them with the metadata. The current line-aware splitter enforces at most 350 BGE tokens per chunk. The earlier splitter could produce 351/352 tokens when a substring was retokenised; those deviations belong to the historical corpus, not this revision. This does not check the 2,000-token evidence budget, which uses the Qwen tokenizer.

**Figure P2-3** separates text and table chunks. **Figure P2-4** shows which documents contain warnings. These warnings concern specific table-chunking issues; they do not mean that all page text has been lost.

Keeping complete lines/words prevents splitting a financial literal, but does not guarantee complete sentences. Overlap repeats some preceding text; it does not ensure that retrieval selects the next chunk or its year/unit header.

In [ ]:
from transformers import AutoTokenizer
bge = AutoTokenizer.from_pretrained(study.CONFIG["embedding"], local_files_only=True)
# Batch in small groups to limit memory on an 8 GB Mac.
actual_lengths = []
for start in range(0, len(chunks), 256):
    encoded = bge([c["text"] for c in chunks[start:start+256]], add_special_tokens=False, truncation=False)
    actual_lengths.extend(len(ids) for ids in encoded["input_ids"])
assert actual_lengths == chunk_df.embedding_tokens.tolist(), "Stored token counts changed"
assert all(x <= study.CONFIG["chunk_tokens"] for x in actual_lengths), "Current chunk exceeds its BGE limit"
summary.update({"token_counts_rechecked": True, "max_embedding_tokens": max(actual_lengths),
    "chunks_above_350": sum(x > 350 for x in actual_lengths)})
chunk_df[chunk_df.embedding_tokens > 350].to_csv(OUT / "chunks_above_target.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, kind in zip(axes, ["text", "table"]):
    data = chunk_df.loc[chunk_df.kind == kind, "embedding_tokens"]
    ax.hist(data, bins=30, color="#42657d"); ax.axvline(350, color="#b87b40", linestyle="--")
    ax.set(title=f"{kind.title()} chunks: {len(data):,}", xlabel="BGE embedding tokens", ylabel="Chunks")
fig.suptitle("P2-3: Chunk lengths — dashed line is the 350-token target")
fig.tight_layout(); fig.savefig(OUT / "P2_3_chunk_lengths.png", dpi=170); plt.show()
print("Maximum:", max(actual_lengths), "| Above target:", summary["chunks_above_350"])
warning_df = pd.DataFrame(warnings)
warning_df["pdf_page"] = warning_df.page0 + 1
warning_df.to_csv(OUT / "parser_warnings.csv", index=False)
display(warning_df.head(20))
print("All warning/admission records saved:", len(warning_df))
fig, ax = plt.subplots(figsize=(9, 3.5))
warning_df.doc_name.value_counts().sort_values().plot.barh(ax=ax, color="#b87b40")
ax.set(title="P2-4: Recorded extraction and table-admission events", xlabel="Warnings", ylabel="Report")
ax.xaxis.get_major_locator().set_params(integer=True)
fig.tight_layout(); fig.savefig(OUT / "P2_4_warnings.png", dpi=170); plt.show()


## Step 4 — Fresh extraction: Re-extract development pages

Use only the 13 unique evidence pages associated with the 10 development questions. For each page, save a source-PDF image, freshly extracted text/tables and comparison results. Passing `fresh_matches_cache` means that the same environment reproduces the cached extraction. It does not establish extraction correctness.

The extraction cell saves PNG images in `development_pages/`, but its table does not display those images. Run the preview cell below to see an actual PDF page inside this notebook. It also prints the absolute paths used in your running environment. Enlarge the page to check years, million/thousand units, negative values in parentheses, headings and merged cells. Do not edit the source PDFs or gold annotations.

In [ ]:
import pdfplumber
page_dir = OUT / "development_pages"; page_dir.mkdir(exist_ok=True)
dev_keys = sorted({(q["doc_name"], e["evidence_page_num"]) for q in dev for e in q["evidence"]})
comparisons = []
for name, page0 in dev_keys:
    path = DATA / "pdfs" / f"{name}.pdf"
    expected = manifest.loc[manifest.doc_name == name, "sha256"].iloc[0]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected
    with pdfplumber.open(path) as pdf:
        page = pdf.pages[page0]
        fresh = {"doc_name":name, "page0":page0, "pdf_page":page0+1,
                 "text":page.extract_text(layout=True) or "", "tables":page.extract_tables()}
        page.to_image(resolution=110).save(page_dir / f"{name}_p{page0+1}.png")
    cached = page_lookup[(name,page0)]
    matched = fresh["text"] == cached["text"] and fresh["tables"] == cached["tables"]
    comparisons.append({"doc_name":name,"page0":page0,"pdf_page":page0+1,
        "fresh_matches_cache":matched,"detected_tables":len(fresh["tables"]),
        "page_chunks":sum(c["doc_name"]==name and c["page0"]==page0 for c in chunks)})
    (page_dir / f"{name}_p{page0+1}.json").write_text(json.dumps(fresh,indent=2))
comparison_df = pd.DataFrame(comparisons)
comparison_df.to_csv(OUT / "development_page_comparison.csv", index=False)
summary.update({"development_questions":len(dev),"development_unique_evidence_pages":len(dev_keys),
    "fresh_matches_cache_pages":int(comparison_df.fresh_matches_cache.sum())})
display(comparison_df)
assert comparison_df.fresh_matches_cache.all(), "Inspect differences before changing parser settings"


### View a source PDF page here

Run the following cell to display Pfizer's PDF viewer page 59. If the PNG does not exist on your machine, the cell renders it from the local PDF first. The displayed image is also saved inside this notebook's output, so you can see it when reopening the executed notebook.

To inspect another audited page, change `preview_report` and `preview_pdf_page` using a row in the table above. Paths printed by the cell refer to the machine running your notebook.

In [ ]:
from pathlib import Path
import pdfplumber
from IPython.display import display, Image

# This preview can run independently of the extraction cell above.
preview_report = "PFIZER_2021_10K"
preview_pdf_page = 59  # PDF viewer page, starting at 1; not the printed page label.
preview_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "rag_study/config.json").is_file())
preview_pdf = preview_root / "rag_study/data/pdfs" / f"{preview_report}.pdf"
preview_dir = preview_root / "rag_study/outputs/data_audit/phase2_line_aware_v2/development_pages"
preview_image = preview_dir / f"{preview_report}_p{preview_pdf_page}.png"
if not preview_pdf.is_file():
    raise FileNotFoundError(f"PDF missing in this environment: {preview_pdf}. Complete Phase 1 first.")
if not preview_image.is_file():
    preview_dir.mkdir(parents=True, exist_ok=True)
    with pdfplumber.open(preview_pdf) as pdf:
        if not 1 <= preview_pdf_page <= len(pdf.pages):
            raise ValueError(f"Choose a PDF page between 1 and {len(pdf.pages)}.")
        pdf.pages[preview_pdf_page - 1].to_image(resolution=110).save(preview_image)
print("PDF in this environment:", preview_pdf.resolve())
print("Page image:", preview_image.resolve())
print(f"Showing {preview_report}, PDF viewer page {preview_pdf_page}.")
display(Image(filename=str(preview_image), width=900))

## Step 5 — Pfizer worked example: Inspect the source page before the extraction

Question: **Did Pfizer grow its PPNE between FY20 and FY21?** PP&E means property, plant and equipment. First locate **2021 / 2020, MILLIONS, and 14,882 / 13,745** on the original page. Then read the extraction.

This is not a new annotation of the answer. The example helps distinguish “the numbers are present” from “the years, units and row labels remain correctly aligned.”

In [ ]:
example = page_lookup[("PFIZER_2021_10K",58)]
display(Image(filename=str(page_dir / "PFIZER_2021_10K_p59.png")))
lines = [line.strip() for line in example["text"].splitlines()
         if any(term in line for term in ["As of December", "MILLIONS", "2021      2020", "Property, plant", "Treasury stock", "comprehensive loss"])]
print("Page text — selected lines:\n" + "\n".join(lines))
print("Detected table — first 12 rows:")
display(pd.DataFrame(example["tables"][0]).head(12))
baseline_chunks = study.read_jsonl(ROOT / "rag_study/outputs/history/before_chunking_repair_20261021/rag_study/data/chunks.jsonl")
example_chunks = [c for c in baseline_chunks if c["doc_name"] == "PFIZER_2021_10K" and c["page0"] == 58]
display(pd.DataFrame([{k:c[k] for k in ["id","kind","embedding_tokens"]} for c in example_chunks]))


## Step 6 — Required facts: Does extraction retain the necessary information together?

**Figure P2-5** only checks whether four groups of strings appear in the Pfizer page extraction. It is not a financial semantic assessment. Green means a string is present; grey means it is missing. Even an entirely green result still requires comparison with the PDF.

**Figure P2-6** compares the original page with the detected table region. Table extraction may retain some numbers while losing years, units and row labels. A text chunk retains necessary information, so the whole page cannot be called an extraction failure. The table cannot be treated as verified either.

**Note:** Chunk 1 also contains 2021/2020 in unrelated text. A green cell does not establish that the amount columns align with those years. The printed label on PDF viewer page 59 of Pfizer is 53. The printed label on PDF viewer page 56 of Best Buy is 54.

**Reading the chunk columns:** `text chunk 0` contains the earlier asset rows, including PP&E, its 2021/2020 values and the unit header. `text chunk 1` contains later liability/equity rows; it need not repeat the PP&E facts. `table chunk 2` is a separate detected-table representation, not a third text fragment; here it loses required headers, row labels and the 2020 column. Chunk numbers indicate fragments, not quality. Chunk 0 preserves the facts needed for this PP&E question, but that does not certify every row: its final liability amount is cut at the boundary and continues in chunk 1.

The first two figures below use **pre-repair chunks**. A missing earlier row in text chunk 1 is normal segmentation. Splitting `25,920` into `25,` is a boundary defect; indexing a table without its labels and value columns is a separate evidence risk. The next comparison shows what the repair changes and what remains unresolved.

In [ ]:
checks = {"Year labels: 2021 + 2020":["2021","2020"],"Unit: MILLIONS":["MILLIONS"],
          "Row: Property, plant and equipment":["Property, plant and equipment"],"Both values: 14,882 + 13,745":["14,882","13,745"]}
representations = {"Page text":example["text"], **{f'{c["kind"]} chunk {j}':c["text"] for j,c in enumerate(example_chunks)}}
presence = pd.DataFrame({name:[all(term in text for term in terms) for terms in checks.values()]
                        for name,text in representations.items()}, index=checks)
presence.to_csv(OUT / "pfizer_fact_presence.csv")
fig, ax = plt.subplots(figsize=(10,4))
from matplotlib.colors import ListedColormap
ax.imshow(presence.values.astype(int), cmap=ListedColormap(["#eeeeee","#6da58a"]),vmin=0,vmax=1,aspect="auto")
ax.set_xticks(range(len(presence.columns)),presence.columns,rotation=20,ha="right")
ax.set_yticks(range(len(presence.index)),presence.index)
for i in range(len(presence.index)):
    for j in range(len(presence.columns)):ax.text(j,i,"present" if presence.iloc[i,j] else "absent",ha="center",va="center")
ax.set_title("P2-5: Pfizer required-fact strings — presence is not correctness")
fig.tight_layout();fig.savefig(OUT / "P2_5_pfizer_fact_presence.png",dpi=170);plt.show()

original = plt.imread(page_dir / "PFIZER_2021_10K_p59.png")
fig, axes = plt.subplots(1,2,figsize=(12,8))
axes[0].imshow(original);axes[0].axis("off");axes[0].set_title("Original PDF: viewer page 59")
axes[1].axis("off");axes[1].set_title("Detected table: first 12 rows only")
table = axes[1].table(cellText=[[str(v or "") for v in row] for row in example["tables"][0][:12]],
                     colLabels=["Extracted cell 1","Extracted cell 2"],loc="upper center")
table.auto_set_font_size(False);table.set_fontsize(11);table.scale(1,1.5)
axes[1].text(0,.35,"Observed: row names, year headers and\nthe 2020 values are missing here.\n\nThe page text and text chunk 0 retain\nthe PP&E row and both values.\n\nAlways compare the source page,\npage text and the selected chunk.",transform=axes[1].transAxes,fontsize=11,va="top")
fig.suptitle("P2-6: A table was detected, but financial structure was lost")
fig.tight_layout();fig.savefig(OUT / "P2_6_pdf_vs_table.png",dpi=180);plt.show()


### Compare the repaired representation

The new splitter keeps complete source lines where possible and complete words otherwise. It may use less than 50 tokens of overlap to avoid splitting a number. A table lacking required labels or explicit source units/year headers is quarantined (kept for inspection, excluded from search). This does not reconstruct lost columns. The graph checks numeric boundaries on 13 development pages; it is not an accuracy score.

In [ ]:
repair_dir = study.OUT / "data_audit/chunking_repair_20261021"
repair = json.loads((repair_dir / "repair_verification.json").read_text())
comparison = pd.read_csv(repair_dir / "development_comparison.csv")
display(comparison)
display(Image(filename=str(ROOT / "Report/assets/overall/Figure_6_Chunking_Repair.png")))
repaired_example = [c for c in chunks if c["doc_name"] == "PFIZER_2021_10K" and c["page0"] == 58]
for c in repaired_example:
    print(c["id"], c["kind"], c["embedding_tokens"], "tokens")
    for line in c["text"].splitlines():
        if "25,920" in line or "Property, plant" in line:
            print("  ", line)
print("Incomplete raw table still retained:", len(example["tables"]), "| Admitted table chunks:", sum(c["kind"] == "table" for c in repaired_example))
print("All nonempty source-text pages retained:", repair["retained_text_pages"], "/", repair["nonempty_text_pages"])
print("Headers may still be on another text chunk. Check sources before scoring support.")

## Step 7 — Audit worksheet and saved results: Record your observations

The code has checked page mappings, hashes, token counts and cached extraction. The user-authorised AI agent has completed the 13-page review, with explicit yes/no/uncertain ratings; completion does not mean all quality checks passed. This is not a relabelling exercise for all 150 questions.

There is one row per development question/page pair. Compare the 13 unique source pages and record issues with units, years, negative values and tables/chunks. One page may support several questions. Existing entries will not be overwritten by this notebook. The completed worksheet records review_method=ai_agent_visual_review and human_reviewed=no; full findings are in agent_review_20261022/.

The completed source-page check remains a historical extraction-quality audit. It does not create an answer-assessment task: later answer scores are generated automatically.


In [ ]:
audit_path = OUT / "development_document_audit.csv"
if not audit_path.exists():
    rows = []
    for q in dev:
        for page0 in sorted({e["evidence_page_num"] for e in q["evidence"]}):
            rows.append({"financebench_id":q["financebench_id"],"doc_name":q["doc_name"],"page0":page0,
                "pdf_page":page0+1,"printed_page_label":"","automatic_page_mapping_ok":True,
                "years_and_columns_ok":"","units_ok":"","negative_values_ok":"","headers_ok":"",
                "merged_cells_ok":"","chunk_supports_required_facts":"","review_status":"pending","notes":""})
    pd.DataFrame(rows).to_csv(audit_path,index=False)
audit = pd.read_csv(audit_path)
summary.update({"checked_at":datetime.now(ZoneInfo("Asia/Hong_Kong")).isoformat(),
    "source_hashes":{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in
                     [DATA/"manifest.csv", DATA/"chunks.jsonl", DATA/"parsing_failures.jsonl", ROOT/"rag_study/study.py"]},
    "audit_rows":len(audit),"document_review_pending_rows":int((audit.review_status == "pending").sum()),
    "manual_document_audit_complete":bool(audit.get("human_reviewed", pd.Series(dtype=str)).eq("yes").sum() == len(audit)),
    "agent_document_audit_complete":bool(audit.review_status.eq("reviewed").all() and audit.get("review_method", pd.Series(dtype=str)).eq("ai_agent_visual_review").sum() == len(audit)),
    "review_methods":sorted(audit.get("review_method", pd.Series(["unrecorded"])).dropna().unique().tolist()),"parser_changed":False,"index_rebuilt":False,"revision":"line_aware_v2",
    "prior_repair_changed_parser_and_rebuilt_index":True,
    "model_calls":0,"final_evaluation_run":False})
(OUT / "phase2_verification.json").write_text(json.dumps(summary,indent=2)+"\n")
print(json.dumps(summary,indent=2))
print("Audit worksheet:",audit_path.relative_to(ROOT))


## Step 8 — Read the AI-assisted visual audit

The original PDF images, extracted JSON and current chunk IDs have now been compared on 13 development pages. **The student authorised AI agent review instead of personal page checking on 22 October 2026. This is recorded as AI review, not human validation.** It checked selected amounts and signs, table/year/unit structure and the opening of the next page; it did not verify every cell or score generated answers.

**P2-7** shows a remaining risk: a later chunk may keep the values but omit their millions/thousands label. Finding such a chunk is not enough to support an amount with units. The source page is still usable; quarantine applies to its incomplete extracted table, not the whole page. Some standalone row fragments could be recovered later using verified headers. No such reconstruction or automatic cross-page stitching is claimed.

Use `selected_page` below to inspect a source page alongside all of its current text chunks. Seeing both chunks in this notebook does not mean both will reach the model: the final evidence allowance remains four whole chunks and 2,000 Qwen tokens. The complete page-by-page issue logs follow the requested four-part audit structure in `visual_audit/Visual_Audit_Report.md`. The warnings shown here are audit records, not a newly enforced runtime abstention rule.

In [ ]:
audit_dir = OUT / "visual_audit"
visual = json.loads((audit_dir / "visual_audit.json").read_text())
assert visual["summary"]["chunks_sha256"] == study.digest(DATA / "chunks.jsonl"), "Visual audit is stale: chunks changed."
assert visual["summary"]["index_signature_sha256"] == study.digest(DATA / "index_signature.json"), "Index revision changed."
display(pd.read_csv(audit_dir / "visual_audit_summary.csv"))
display(Image(filename=str(audit_dir / "P2_7_unit_context.png")))
selected_page = ("PFIZER_2021_10K", 57)  # Change to any document/page in the table.
review = next(p for p in visual["pages"] if (p["doc_name"], p["pdf_page"]) == selected_page)
assert study.digest(ROOT / review["source_pdf"]) == review["source_pdf_sha256"]
assert study.digest(ROOT / review["source_image"]) == review["source_image_sha256"]
display(Image(filename=str(ROOT / review["source_image"])))
print("Quality:", review["overall_quality"], "| Quarantine:", review["quarantine_decision"])
print("Continuity:", review["page_continuity"])
display(pd.DataFrame(review["issues"]))
assert study.digest(ROOT / review["extracted_page_json"]) == review["extracted_page_json_sha256"]
raw_page = json.loads((ROOT / review["extracted_page_json"]).read_text())
print("Extracted page text:", raw_page["text"].strip())
print("Raw detected tables:", len(raw_page["tables"]))
raw_table_index = 0  # Change to inspect another detected object; none are admitted here.
if raw_page["tables"]:
    display(pd.DataFrame(raw_page["tables"][raw_table_index]))
for chunk in chunks:
    if chunk["id"] in review["chunk_ids"]:
        print("\n", chunk["id"], "|", chunk["embedding_tokens"], "BGE tokens")
        print(chunk["text"].strip())
print("Full audit:", audit_dir / "Visual_Audit_Report.md")
print("AI extraction review completed; author subsequently confirmed checking all 13 pages. Answer scoring remains separate.")

print("Completed delegated review (22 October):")
display(audit[["doc_name", "pdf_page", "years_and_columns_ok", "units_ok", "headers_ok", "chunk_supports_required_facts", "review_status", "review_method", "human_reviewed"]])
print("Fresh agent findings:", OUT / "agent_review_20261022/agent_review.md")

## Exercises and stopping point

1. Why does Pfizer's `extract_tables()` returning one table not establish that the table is correct?
2. Why might text chunk 1 contain amounts and negative values in parentheses without making their years clear?
3. Which tokenizers count the 350-token target and the 2,000-token budget?

**Answer guide:** Detecting a table does not guarantee that rows, columns and units are preserved. A chunk may lose the years or units at the top of its source page. The 350-token target uses BGE; the 2,000-token final evidence limit uses Qwen.

Next, choose the Best Buy question at `dev[3]`. Compare revenue, net earnings, years and units in `development_pages/BESTBUY_2017_10K_p56.png` and its corresponding JSON file. Record parser quality only; do not run an answer comparison.

### What can go into the report?

- **2.2 Dataset selection and document audit:** State that the actual audit scope is 10 development questions and 13 unique evidence pages. Describe programmatic comparison separately from the student's visual audit.
- **2.3 Processing and retrieval:** Report the 4,900 pages, text/table chunk counts, pages with no extracted text, warnings and token-length deviations. These are processing observations, not accuracy results.
- **Implementation/difficulties in the progress report:** Use P2-6 to explain the loss of labels, years and units in the Pfizer example. P2-3 can illustrate chunking diagnostics. Keep extra figures in the appendix or notebook if space is limited; do not insert all six into a seven-page proposal.
- Claim year/unit preservation only for examples that have been compared with the source. Do not claim that all financial content in the 26 reports is correct.

**Stop:** After the programmatic checks, read the figures and complete the worksheet. Summarise unresolved quality issues, then confirm before Phase 3. This lesson has not changed the parser or index.